# SPADE Demo — EGFR (P00533) + Erlotinib

This notebook walks through the full SPADE pipeline end-to-end:

1. **Install** dependencies (Colab-compatible)
2. **Fetch** EGFR structure from EBI AlphaFold DB
3. **Flexibility analysis** — pLDDT tiers + PAE-weighted graph
4. **Ensemble generation** — PAE-weighted NMA conformers
5. **Ligand preparation** — erlotinib SMILES → stereoisomers → PDBQT
6. **Ensemble docking** — Vina per conformer with per-conformer bounding box
7. **Pose clustering** — PLIF fingerprints + DBSCAN consensus
8. **Report generation** — provenance.json + HTML report

In [ ]:
# ── Install (run once on Colab; skip if already installed) ────────────────────
import sys

IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    !pip install -q spade-docking  # install from PyPI once published
    # Or install from source:
    # !pip install -q git+https://github.com/ChandraguptSharma07/Spade.git
    !pip install -q meeko dimorphite-dl prolif scikit-learn jinja2 rich questionary typer
    !conda install -q -c conda-forge prody vina -y  # requires conda kernel

print('Dependencies ready.')

## 1. Fetch EGFR structure

EGFR (Epidermal Growth Factor Receptor, UniProt P00533) is a well-characterised kinase target with a known erlotinib binding site in the ATP pocket.

`fetch_structure()` downloads the AlphaFold v4 PDB and PAE JSON from the EBI AlphaFold DB.

In [ ]:
from spade.core.structure import fetch_structure

structure = fetch_structure('P00533')

print(f'UniProt:   {structure.uniprot_id}')
print(f'AF version: {structure.af_version}')
print(f'Residues:  {structure.n_residues}')
print(f'Mean pLDDT: {structure.plddt.mean():.1f}')
print(f'PAE matrix: {structure.pae_matrix.shape}')

## 2. Flexibility analysis

We classify every residue by its pLDDT confidence tier and build a PAE-weighted flexibility graph restricted to the ATP pocket neighbourhood (12 Å cutoff).

**Critical:** inter-domain PAE values (residues far from the pocket) are zeroed out regardless of their PAE value. A flexible domain on the other side of the protein must not contaminate the local docking ensemble.

In [ ]:
import numpy as np
from spade.core.flexibility import classify_residues, build_flexibility_profile

# Classify all residues by pLDDT tier
tiers = classify_residues(structure.plddt)
from collections import Counter
tier_counts = Counter(tiers.values())
print('pLDDT tier distribution:')
for tier in ['rigid', 'moderate', 'flexible', 'disordered']:
    n = tier_counts.get(tier, 0)
    print(f'  {tier:12s}: {n:4d} ({n/structure.n_residues*100:.1f}%)')

# Define the ATP pocket: residues 695–720 (EGFR kinase hinge, 0-based indices)
pocket_residues = np.arange(695, 720)

ca_sel = structure.atoms.select('name CA')
ca_coords = ca_sel.getCoords()

profile = build_flexibility_profile(
    structure.plddt,
    structure.pae_matrix,
    pocket_residues,
    ca_coords,
)

print(f'\nPocket residues: {len(profile.pocket_residues)}')
print(f'Disordered residues: {len(profile.disordered_residues)}')
print(f'Inter-domain PAE warning: {profile.inter_domain_pae_warning}')
print(f'Mode weight vector shape: {profile.mode_weight_vector.shape}')

## 3. Ensemble generation

`EnsembleGenerator` builds an ANM on pocket-local Cα atoms, weights modes by the PAE flexibility profile, displaces along those modes at varied amplitudes, and hard-rejects any conformer with Cα RMSD > 1.2 Å from the reference.

Side-chain clashes introduced by backbone displacement are resolved by the Dunbrack repacker.

In [ ]:
from spade.core.ensemble import EnsembleGenerator, MAX_CA_RMSD_ANGSTROM

gen = EnsembleGenerator(structure, profile, n_conformers=10)
conformers = gen.generate()

print(f'Generated {len(conformers)} conformers')
print(f'Cα RMSD from reference (must be ≤ {MAX_CA_RMSD_ANGSTROM} Å):')
for i, conf in enumerate(conformers):
    rmsd = float(conf.getData('ca_rmsd_from_ref')[0])
    bar = '█' * int(rmsd / MAX_CA_RMSD_ANGSTROM * 20)
    print(f'  conf {i:2d}: {rmsd:.3f} Å  {bar}')

## 4. Ligand preparation

Erlotinib has no stereocenters, so `prepare_ligand` returns one variant per tautomer. The pipeline:
1. Enumerate tautomers (top 3 by RDKit)
2. Enumerate protomers at pH 7.4 (Dimorphite-DL)
3. Detect undefined stereocenters — erlotinib has none
4. Generate 3D conformer with ETKDGv3 + MMFF
5. Prepare Meeko PDBQT

In [ ]:
from spade.core.ligand import prepare_ligand

ERLOTINIB_SMILES = 'CCOc1cc2ncnc(Nc3cccc(C#C)c3)c2cc1OCCOCCO'

ligands = prepare_ligand(ERLOTINIB_SMILES, ph=7.4, enumerate_stereo=True)

print(f'Prepared {len(ligands)} ligand variant(s)')
for i, lig in enumerate(ligands):
    print(f'  [{i}] tautomer={lig.tautomer_id} stereo={lig.stereoisomer_id} '
          f'undef_centers={lig.n_undefined_stereocenters} '
          f'pdbqt_len={len(lig.pdbqt_string)} chars')

## 5. Ensemble docking

`dock_ensemble` iterates over every (conformer, ligand) pair. For each conformer it **recomputes the bounding box** from that conformer's pocket Cα centroid — the centroid shifts by 1–2 Å with NMA displacement, so a static box from the reference structure would score the wrong region.

**Note:** AutoDock Vina must be installed (`conda install -c conda-forge vina`). If it's not available, the cell below will raise an ImportError.

In [ ]:
from spade.core.docking import dock_ensemble

docking_results = dock_ensemble(
    conformers,
    ligands,
    pocket_residues,
    exhaustiveness=8,
    n_poses=9,
)

print(f'Total docking runs: {len(docking_results)}')
for dr in docking_results[:3]:  # show first 3
    best = min((p.score_kcal_mol for p in dr.poses), default=float('nan'))
    print(f'  conf={dr.conformer_index} bbox_center={dr.bounding_box.center.round(2)} '
          f'best_score={best:.2f} kcal/mol time={dr.docking_time_seconds:.1f}s')

## 6. Pose clustering

Poses are clustered by **PLIF interaction fingerprint** similarity (Tanimoto), not by Cartesian RMSD. RMSD between poses from different conformers is mathematically undefined — their coordinate frames are different. PLIF fingerprints capture pharmacophoric equivalence and are conformer-agnostic.

In [ ]:
from spade.core.clustering import cluster_poses

consensus = cluster_poses(
    docking_results,
    conformers,
    ligands[0].mol,
    similarity_threshold=0.7,
)

top = consensus.top_cluster
print(f'Total poses:          {consensus.n_total_poses}')
print(f'Clusters found:       {consensus.n_clusters}')
print(f'Pose confidence:      {consensus.pose_confidence*100:.1f}%')
print(f'Site confidence:      {consensus.site_confidence}')
print()
print('Top cluster:')
print(f'  Mean score:         {top.mean_score:.2f} ± {top.score_std:.2f} kcal/mol')
print(f'  Consensus score:    {top.consensus_score:.2f} kcal/mol')
print(f'  Conformer coverage: {top.n_conformers_represented}/{len(conformers)} '
      f'({top.fraction_ensemble*100:.0f}%)')
print(f'  Poses in cluster:   {len(top.member_poses)}')

## 7. Report generation

Generate `provenance.json` (all pipeline metadata) and `report.html` (Jinja2-rendered summary with score table and embedded provenance).

In [ ]:
import json
from pathlib import Path
from spade.core.report import RunProvenance, ConformerSummary, generate_report

conf_summaries = []
for dr in docking_results:
    scores = [p.score_kcal_mol for p in dr.poses] if dr.poses else [0.0]
    conf_summaries.append(ConformerSummary(
        conformer_index=dr.conformer_index,
        ca_rmsd_from_ref=dr.conformer_ca_rmsd,
        n_poses=len(dr.poses),
        best_score_kcal_mol=min(scores),
        docking_time_seconds=dr.docking_time_seconds,
    ))

prov = RunProvenance(
    uniprot_id='P00533',
    af_version=structure.af_version,
    n_residues=structure.n_residues,
    pocket_residues=pocket_residues.tolist(),
    ligand_smiles=ERLOTINIB_SMILES,
    n_ligand_variants=len(ligands),
    n_conformers_requested=10,
    n_conformers_generated=len(conformers),
    exhaustiveness=8,
    n_poses_per_conformer=9,
    n_total_poses=consensus.n_total_poses,
    n_clusters=consensus.n_clusters,
    top_cluster_score=top.mean_score,
    top_cluster_fraction_ensemble=top.fraction_ensemble,
    site_confidence=consensus.site_confidence,
    conformer_summaries=conf_summaries,
    plddt_mean=float(structure.plddt.mean()),
    plddt_std=float(structure.plddt.std()),
    inter_domain_pae_warning=profile.inter_domain_pae_warning,
)

output_dir = './spade_egfr_erlotinib'
generate_report(prov, output_dir)

out = Path(output_dir)
print(f'Files written to {output_dir}:')
for f in out.iterdir():
    print(f'  {f.name}  ({f.stat().st_size} bytes)')

# Preview the provenance JSON
with open(out / 'provenance.json') as fp:
    pj = json.load(fp)
print(f'\nprovenance.json has {len(pj)} top-level keys')
print('run_id:', pj['run_id'])

## 8. Inline HTML preview (Jupyter only)

Display the generated HTML report inline. On Colab, the py3Dmol viewer will render the binding pocket interactively if 3D coordinates are embedded.

In [ ]:
from IPython.display import IFrame, display

display(IFrame(src='./spade_egfr_erlotinib/report.html', width='100%', height=600))

---
## Summary

| Step | SPADE function |
|------|----------------|
| Fetch structure | `fetch_structure(uniprot_id)` |
| Flexibility profile | `build_flexibility_profile(plddt, pae, pocket, coords)` |
| Ensemble | `EnsembleGenerator(structure, profile).generate()` |
| Ligand prep | `prepare_ligand(smiles)` |
| Docking | `dock_ensemble(conformers, ligands, pocket)` |
| Clustering | `cluster_poses(docking_results, conformers, mol)` |
| Report | `generate_report(provenance, output_dir)` |

The `spade run` CLI command runs all seven steps with a single command:

```bash
spade run --uniprot P00533 --ligand 'CCOc1cc2ncnc(Nc3cccc(C#C)c3)c2cc1OCCOCCO' --output ./out
```